# 008 - Level8

## 8. Set Operations & Reshaping

### 🔄 Operaciones de conjuntos y remodelado de datos

- **UNION / UNION ALL**  
  Permiten **apilar tablas verticalmente** (concatenación vertical).
  - `UNION` elimina duplicados.
  - `UNION ALL` conserva todos los registros.

- **PIVOT / UNPIVOT**  
  Rotan los datos:
  - De **filas a columnas** (`PIVOT`)
  - De **columnas a filas** (`UNPIVOT`)  
  Estas operaciones se conocen como **reshaping**.

- **Melt / Wide-to-Long**  
  Transformaciones esenciales para:
  - **Visualización de datos**
  - **Reporting**


In [2]:
import pandas as pd
import numpy as np
import polars as pl

In [3]:
df_matches = pd.read_csv('../Data/WorldCupMatches.csv').rename(columns=lambda col: col.lower())
df_players = pd.read_csv('../Data/WorldCupPlayers.csv').rename(columns=lambda col: col.lower())
df_cups = pd.read_csv('../Data/WorldCups.csv').rename(columns=lambda col: col.lower())

pl_matches = pl.read_csv('../Data/WorldCupMatches.csv').rename(lambda col: col.lower())
pl_players = pl.read_csv('../Data/WorldCupPlayers.csv').rename(lambda col: col.lower())
pl_cups = pl.read_csv('../Data/WorldCups.csv').rename(lambda col: col.lower())

## 🏆 Ejercicio 8.1: Consolidación de Estadísticas Globales

**Nivel:** 8 (Set Operations)  
**Técnica:** Unión Vertical (Concatenación)

### 🎯 Objetivo
Calcular el **total de goles históricos por país**, unificando los registros donde el equipo actuó como **local** y como **visitante**.

---

### 📋 El Problema
En el dataset `worldcup.matches`, los datos de desempeño están divididos en dos columnas:

- `Home Team Name`
- `Away Team Name`

Si contamos los goles usando solo una de estas columnas, obtendremos una **visión parcial** (solo el 50 % de la realidad).

Para construir el **ranking histórico real de goles por país**, es necesario:

- **Apilar verticalmente** ambas dimensiones (local y visitante)
- Unificarlas en una **única estructura**
- Luego, **agregar (SUM)** los goles por selección


```SQL
WITH todos_los_goles AS (
    SELECT
        "Home Team Name" AS equipo,
        "Home Team Goals" AS goles
    FROM worldcup.matches
    UNION ALL
    SELECT
        "Away Team Name" AS equipo,
        "Away Team Goals" AS goles
    FROM worldcup.matches
)
SELECT equipo, SUM(goles) AS total_goles
FROM todos_los_goles
GROUP BY equipo
ORDER BY total_goles DESC;
```

In [11]:
df_m = df_matches.copy()

home = df_m[['home team name', 'home team goals']].rename(
    columns={'home team name': 'team', 'home team goals': 'goals'}
)

away = df_m[['away team name', 'away team goals']].rename(
    columns={'away team name': 'team', 'away team goals': 'goals'}
)

df_total = pd.concat([home,away], ignore_index=True)

ranking_goles = df_total.groupby('team')['goals'].sum().sort_values(ascending=False).reset_index()

In [17]:
home = pl_matches.with_columns([
    pl.col('home team name').alias('team'),
    pl.col('home team goals').alias('goals')
])

away = pl_matches.with_columns([
    pl.col('away team name').alias('team'),
    pl.col('away team goals').alias('goals')
])

df_total = pl.concat([home, away])

ranking_goles = (
    df_total.group_by('team')
    .agg(pl.col('goals').sum().alias('total_goals'))
    .sort('total_goals', descending=True)
)

## 🏆 Ejercicio 8.2: Reshaping (PIVOT) – Evolución por Décadas

**Nivel:** 8 (Reshaping)  
**Técnica:** Pivot Table (de formato **Largo** a **Ancho**)

### 🎯 Objetivo
Crear una **matriz comparativa** que muestre los **goles totales** de los **“Big 4”**  
(Brasil, Argentina, Alemania e Italia) **divididos por décadas**.

---

### 📋 El Problema
Los datos actuales están en formato **Long** (una fila por partido).

Si queremos comparar el desempeño histórico de las grandes potencias mundiales, una lista larga resulta **difícil de leer e interpretar**.  
Para obtener una visión clara y comparativa, necesitamos:

- Transformar los **Años / Décadas** en **columnas**
- Mantener a los **países como filas**
- Construir una **tabla tipo matriz** que nos permita ver la “fotografía” completa de la evolución futbolística a lo largo del tiempo


```SQL
SELECT
    equipo,
    SUM(CASE WHEN year BETWEEN 1930 AND 1959 THEN goles ELSE 0 END) AS "Era_Temprana",
    SUM(CASE WHEN year BETWEEN 1960 AND 1989 THEN goles ELSE 0 END) AS "Era_Dorada",
    SUM(CASE WHEN year >= 1990 THEN goles ELSE 0 END) AS "Era_Moderna"
FROM (
    SELECT year AS year, "Home Team Name" AS equipo, "Home Team Goals" AS goles FROM worldcup.matches
    UNION ALL
    SELECT year AS year, "Away Team Name" AS equipo, "Away Team Goals" AS goles FROM worldcup.matches
) AS subquery
WHERE equipo in ('Brazil', 'Argentina', 'Germany', 'Italy')
GROUP BY equipo;
```

In [27]:
home = df_matches[['year', 'home team name', 'home team goals']].rename(
    columns={'home team name': 'equipo', 'home team goals': 'goles'}
)

away = df_matches[['year', 'away team name', 'away team goals']].rename(
    columns={'away team name': 'equipo', 'away team goals': 'goles'}
)

df_total = pd.concat([home, away], ignore_index=True)

def categorizar_era(y):
    if 1930 <= y <= 1959: return 'Era_Temprana'
    if 1960 <= y <= 1989: return 'Era_Dorada'
    return 'Era_Moderna'

df_total['era'] = df_total['year'].apply(categorizar_era)

resultado_pd = df_total.pivot_table(
    index='equipo',
    columns='era',
    values='goles',
    aggfunc='sum',
    fill_value = 0
)

big_4 = ['Brazil', 'Argentina', 'Germany', 'Italy']

# resultado_pd.reindex(big_4)

df_para_unpivot = resultado_pd.reset_index()

df_unpivot = df_para_unpivot.melt(
    id_vars=['equipo'],
    value_vars=['Era_Dorada', 'Era_Moderna', 'Era_Temprana'],
    var_name = 'era_nombre',
    value_name = 'total_goles'
)

df_unpivot[df_unpivot['equipo'].isin(big_4)]


,equipo,era_nombre,total_goles
2,Argentina,Era_Dorada,52.0
7,Brazil,Era_Dorada,78.0
28,Germany,Era_Dorada,0.0
39,Italy,Era_Dorada,46.0
85,Argentina,Era_Moderna,56.0
90,Brazil,Era_Moderna,81.0
111,Germany,Era_Moderna,90.0
122,Italy,Era_Moderna,49.0
168,Argentina,Era_Temprana,25.0
173,Brazil,Era_Temprana,66.0


In [28]:
home_pl = pl_matches.select([
    pl.col('year'),
    pl.col('home team name').alias('equipo'),
    pl.col('home team goals').alias('goles')
])

away_pl = pl_matches.select([
    pl.col('year'),
    pl.col('away team name').alias('equipo'),
    pl.col('away team goals').alias('goles')
])

df_total_pl = pl.concat([home_pl, away_pl]).with_columns(
    pl.when(pl.col('year') <= 1959).then(pl.lit('Era_Temprana'))
    .when(pl.col('year') <1989).then(pl.lit('Era_Dorada'))
    .otherwise(pl.lit('Era_Moderna'))
    .alias('era')
)

resultado_pl = df_total_pl.pivot(
    on='era',
    index='equipo',
    values='goles',
    aggregate_function='sum'
).filter(pl.col('equipo').is_in(['Brazil','Argentina','Germany','Italy'])).fill_null(0)

#resultado_pl

df_unpivot_pl = resultado_pl.unpivot(
    index='equipo',
    on=['Era_Temprana', 'Era_Dorada', 'Era_Moderna'],
    variable_name = 'era_nombre',
    value_name = 'total_goels'
)

df_unpivot_pl

equipo,era_nombre,total_goels
str,str,i64
"""Argentina""","""Era_Temprana""",25
"""Brazil""","""Era_Temprana""",66
"""Germany""","""Era_Temprana""",14
"""Italy""","""Era_Temprana""",33
"""Argentina""","""Era_Dorada""",52
…,…,…
"""Italy""","""Era_Dorada""",46
"""Argentina""","""Era_Moderna""",56
"""Brazil""","""Era_Moderna""",81
